In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import h

K = 2        # genes per panel

In [2]:
TEST_SIZE = 0.2      # 80% train, 20% test

adata_mel = h.prepare(h.load_melanoma())
train_mel, test_mel = h.split(adata_mel, test_size=TEST_SIZE)
print(adata_mel.n_obs, "cells |", len(train_mel), "train |", len(test_mel), "test")

4513 cells | 3610 train | 903 test


### Candidate genes: four ways
- markers_hvg: highly variable genes + hand-picked markers (my first way)
- wilcoxon: Wilcoxon test on all genes, no hand-picking
- lasso: machine learning (L1 logistic regression) picks the genes
- stable: Wilcoxon repeated on many random halves of the training cells; keep the most frequent winners
All use the training cells only.

In [3]:
POOLS = ["markers_hvg", "wilcoxon", "lasso", "stable"]

cands_mel = {}
for pool in POOLS:
    cands_mel[pool] = h.candidates(adata_mel, train_mel, method=pool, n_boot=50)
    print(f"{pool:12s}", cands_mel[pool])

markers_hvg  [np.str_('SPARC'), np.str_('APOD'), np.str_('CALD1'), np.str_('PMEL'), np.str_('TIMP1'), np.str_('MLANA'), np.str_('CTSL1'), np.str_('CCND1'), np.str_('PTPRC'), np.str_('CD74'), np.str_('CD3E'), np.str_('LYZ'), np.str_('UGDH-AS1'), np.str_('CABP4'), np.str_('SHISA9'), np.str_('CCL22')]
wilcoxon     ['S100B', 'SERPINE2', 'CD63', 'PLP1', 'VAT1', 'LGALS3BP', 'GPNMB', 'PRAME', 'TMSB4X', 'PTPRC', 'ARHGDIB', 'LAPTM5', 'IKZF3', 'STK17B', 'CXCR4', 'B2M']
lasso        ['PLP1', 'SERPINE2', 'PRAME', 'SOX10', 'CDH19', 'S100B', 'EDNRB', 'GPR143', 'TMSB4X', 'B2M', 'FBLIM1', 'ARHGDIB', 'CD74', 'COL1A1', 'CXCL12', 'HLA-C']
stable       ['S100B', 'SERPINE2', 'CD63', 'VAT1', 'PLP1', 'LGALS3BP', 'GPNMB', 'PRAME', 'TMSB4X', 'PTPRC', 'ARHGDIB', 'LAPTM5', 'IKZF3', 'STK17B', 'CXCR4', 'B2M']


In [4]:
qubo_mel = {}
for pool, cands in cands_mel.items():
    alpha, cv = h.choose_alpha_cv(adata_mel, train_mel, cands, K)
    Q, const, info = h.build_qubo(adata_mel, train_mel, cands, K, alpha, "auto")
    panel = h.best_panel(Q, const, cands, K)
    qubo_mel[pool] = {"Q": Q, "const": const, "info": info, "alpha": alpha, "panel": panel}
    print(f"{pool:12s} alpha={alpha:<5} lambda={info['lam']:.3f}  QUBO pick: {panel}")

markers_hvg  alpha=0.05  lambda=0.854  QUBO pick: [np.str_('APOD'), np.str_('PTPRC')]
wilcoxon     alpha=0.0   lambda=0.957  QUBO pick: ['S100B', 'TMSB4X']
lasso        alpha=0.0   lambda=0.957  QUBO pick: ['S100B', 'TMSB4X']
stable       alpha=0.0   lambda=0.957  QUBO pick: ['S100B', 'TMSB4X']


In [5]:
lasso_mel = h.lasso_panel(adata_mel, train_mel, K)    # ML rival, picks from all genes

rows = []
for pool, q in qubo_mel.items():
    cands, r = cands_mel[pool], q["info"]["r"]
    panels = {"classical: top 2": [cands[i] for i in np.argsort(-r)[:K]],
              "QUBO (exact)": q["panel"]}
    for method, p in panels.items():
        m, s = h.repeated_splits(adata_mel, p, test_size=TEST_SIZE)
        rows.append({"pool": pool, "method": method, "panel": ", ".join(map(str, p)),
                     "test": round(h.evaluate(adata_mel, p, train_mel, test_mel), 3),
                     "10 splits": f"{m:.3f} ± {s:.3f}"})

m, s = h.repeated_splits(adata_mel, lasso_mel, test_size=TEST_SIZE)
rows.append({"pool": "all genes", "method": "classical: lasso", "panel": ", ".join(lasso_mel),
             "test": round(h.evaluate(adata_mel, lasso_mel, train_mel, test_mel), 3),
             "10 splits": f"{m:.3f} ± {s:.3f}"})

acc_mel = pd.DataFrame(rows)
acc_mel

,pool,method,panel,test,10 splits
0,markers_hvg,classical: top 2,"APOD, SPARC",0.919,0.927 ± 0.009
1,markers_hvg,QUBO (exact),"APOD, PTPRC",0.954,0.951 ± 0.006
2,wilcoxon,classical: top 2,"TMSB4X, S100B",0.980,0.982 ± 0.004
3,wilcoxon,QUBO (exact),"S100B, TMSB4X",0.980,0.982 ± 0.004
4,lasso,classical: top 2,"TMSB4X, S100B",0.980,0.982 ± 0.004
5,lasso,QUBO (exact),"S100B, TMSB4X",0.980,0.982 ± 0.004
6,stable,classical: top 2,"TMSB4X, S100B",0.980,0.982 ± 0.004
7,stable,QUBO (exact),"S100B, TMSB4X",0.980,0.982 ± 0.004
8,all genes,classical: lasso,"TMSB4X, S100B",0.980,0.982 ± 0.004


### QAOA (stable pool)
Same QUBO, now with the smallest safe lambda (about 0.96 instead of 2).
Standard X mixer: p = 1, 2, 3. XY mixer (always keeps exactly 2 genes): p = 1, 2.
Each p starts from the previous best angles (INTERP) plus random restarts.

In [6]:
q = qubo_mel["stable"]
E_mel = h.all_energies(q["Q"], q["const"])

base = h.baseline_row(E_mel, k=K)
tab_x, res_x = h.qaoa_ladder(E_mel, p_max=3, k=K)
tab_xy, res_xy = h.qaoa_ladder(E_mel, p_max=2, mixer="xy", init="dicke", k=K)

print("random guess:", {k_: round(v, 5) for k_, v in base.items()})
print("\nX mixer");  display(tab_x.round(5))
print("XY mixer");   display(tab_xy.round(5))


random guess: {'avg energy': 32.07696, 'approx ratio': np.float64(0.80856), 'P(best panel)': 2e-05, 'P(k genes)': 0.00183, 'shots for 99%': 301803}

X mixer


,avg energy,approx ratio,P(best panel),P(k genes),shots for 99%,seconds
p,,,,,,
1,-0.43120,0.99229,0.00297,0.34681,1550,6.48402
2,-1.01938,0.99562,0.00368,0.42180,1250,53.74387
3,-1.20845,0.99668,0.00406,0.44704,1132,78.37836


XY mixer


,avg energy,approx ratio,P(best panel),P(k genes),shots for 99%,seconds
p,,,,,,
1,-1.56018,0.99867,0.01074,1.0,427,17.10557
2,-1.57459,0.99875,0.01416,1.0,323,31.24303


In [7]:
Q2, c2, _ = h.build_qubo(adata_mel, train_mel, cands_mel["stable"], K, q["alpha"], lam=2.0)
tab_x_old, _ = h.qaoa_ladder(h.all_energies(Q2, c2), p_max=2, k=K)
tab_x_old.round(5)

,avg energy,approx ratio,P(best panel),P(k genes),shots for 99%,seconds
p,,,,,,
1,1.04569,0.99255,0.00329,0.38932,1399,6.91808
2,-0.22933,0.99589,0.00395,0.46262,1165,51.07000


In [8]:
results = {}

## 2. Head and neck

In [9]:
hn = results["head and neck"] = {}
hn["adata"] = h.prepare(h.load_hnscc())
hn["train"], hn["test"] = h.split(hn["adata"], test_size=TEST_SIZE)
print(hn["adata"].n_obs, "cells |", len(hn["train"]), "train |", len(hn["test"]), "test")

5578 cells | 4462 train | 1116 test


In [10]:
results["melanoma"] = {
    "adata": adata_mel, "train": train_mel, "test": test_mel,
    "cands": cands_mel, "qubo": qubo_mel, "acc": acc_mel,
    "qaoa_x": tab_x, "qaoa_xy": tab_xy,
}
print(results["melanoma"]["adata"].n_obs, "cells,", len(results["melanoma"]["acc"]), "result rows")

4513 cells, 9 result rows


In [11]:
hn["cands"] = {}
for pool in POOLS:
    hn["cands"][pool] = [str(g) for g in h.candidates(hn["adata"], hn["train"], method=pool, n_boot=50)]
    print(f"{pool:12s}", hn["cands"][pool])

markers_hvg  ['KRT5', 'KRT17', 'KRT19', 'KRT14', 'KRT18', 'ESRG', 'KRT6C', 'LYPD3', 'ADAMTS4', 'LYZ', 'SLC4A8', 'MAP1LC3C', 'PCDH11X', 'IL12RB1', 'CCL22', 'PTPRC']
wilcoxon     ['KRT5', 'KRT17', 'PERP', 'FXYD3', 'DSP', 'GSTP1', 'MIR205HG', 'SFN', 'MALAT1', 'IFITM2', 'VIM', 'B2M', 'TSC22D3', 'SPN', 'TMSB4X', 'IKZF3']
lasso        ['PERP', 'MIR205HG', 'KRT5', 'SFN', 'SPINT2', 'S100A2', 'KRT8', 'DSP', 'IFITM2', 'TSC22D3', 'VIM', 'B2M', 'MALAT1', 'TIMP1', 'PIP4K2A', 'ZFP36']
stable       ['KRT5', 'KRT17', 'PERP', 'FXYD3', 'DSP', 'GSTP1', 'MIR205HG', 'SFN', 'MALAT1', 'IFITM2', 'B2M', 'VIM', 'TSC22D3', 'TMSB4X', 'SPN', 'IKZF3']


In [12]:
hn["qubo"] = {}
for pool, cands in hn["cands"].items():
    alpha, cv = h.choose_alpha_cv(hn["adata"], hn["train"], cands, K)
    Q, const, info = h.build_qubo(hn["adata"], hn["train"], cands, K, alpha, "auto")
    panel = h.best_panel(Q, const, cands, K)
    hn["qubo"][pool] = {"Q": Q, "const": const, "info": info, "alpha": alpha, "panel": panel}
    print(f"{pool:12s} alpha={alpha:<5} lambda={info['lam']:.3f}  QUBO pick: {panel}")

markers_hvg  alpha=0.0   lambda=1.012  QUBO pick: ['KRT5', 'KRT17']
wilcoxon     alpha=0.0   lambda=1.012  QUBO pick: ['KRT5', 'KRT17']
lasso        alpha=0.0   lambda=1.012  QUBO pick: ['KRT5', 'DSP']
stable       alpha=0.0   lambda=1.012  QUBO pick: ['KRT5', 'KRT17']


In [13]:
hn["lasso"] = h.lasso_panel(hn["adata"], hn["train"], K)

rows = []
for pool, q in hn["qubo"].items():
    cands, r = hn["cands"][pool], q["info"]["r"]
    panels = {"classical: top 2": [cands[i] for i in np.argsort(-r)[:K]],
              "QUBO (exact)": q["panel"]}
    for method, p in panels.items():
        m, s = h.repeated_splits(hn["adata"], p, test_size=TEST_SIZE)
        rows.append({"pool": pool, "method": method, "panel": ", ".join(map(str, p)),
                     "test": round(h.evaluate(hn["adata"], p, hn["train"], hn["test"]), 3),
                     "10 splits": f"{m:.3f} ± {s:.3f}"})

m, s = h.repeated_splits(hn["adata"], hn["lasso"], test_size=TEST_SIZE)
rows.append({"pool": "all genes", "method": "classical: lasso", "panel": ", ".join(hn["lasso"]),
             "test": round(h.evaluate(hn["adata"], hn["lasso"], hn["train"], hn["test"]), 3),
             "10 splits": f"{m:.3f} ± {s:.3f}"})

hn["acc"] = pd.DataFrame(rows)
hn["acc"]

,pool,method,panel,test,10 splits
0,markers_hvg,classical: top 2,"KRT5, KRT17",0.992,0.993 ± 0.001
1,markers_hvg,QUBO (exact),"KRT5, KRT17",0.992,0.993 ± 0.001
2,wilcoxon,classical: top 2,"KRT5, KRT17",0.992,0.993 ± 0.001
3,wilcoxon,QUBO (exact),"KRT5, KRT17",0.992,0.993 ± 0.001
4,lasso,classical: top 2,"KRT5, DSP",0.987,0.990 ± 0.002
5,lasso,QUBO (exact),"KRT5, DSP",0.987,0.990 ± 0.002
6,stable,classical: top 2,"KRT5, KRT17",0.992,0.993 ± 0.001
7,stable,QUBO (exact),"KRT5, KRT17",0.992,0.993 ± 0.001
8,all genes,classical: lasso,"KRT5, FXYD3",0.988,0.991 ± 0.002


### QAOA (stable pool)
X mixer: p = 1, 2, 3. XY mixer (always keeps exactly 2 genes): p = 1, 2.

In [14]:
q = hn["qubo"]["stable"]
hn["E"] = h.all_energies(q["Q"], q["const"])

hn["qaoa_x"], _ = h.qaoa_ladder(hn["E"], p_max=3, k=K)
hn["qaoa_xy"], _ = h.qaoa_ladder(hn["E"], p_max=2, mixer="xy", init="dicke", k=K)

print("X mixer");  display(hn["qaoa_x"].round(5))
print("XY mixer"); display(hn["qaoa_xy"].round(5))

X mixer


,avg energy,approx ratio,P(best panel),P(k genes),shots for 99%,seconds
p,,,,,,
1,-0.13986,0.99078,0.00318,0.36130,1448,6.68040
2,-0.77854,0.99416,0.00402,0.43875,1145,53.78587
3,-0.98513,0.99525,0.00471,0.47486,976,78.86801


XY mixer


,avg energy,approx ratio,P(best panel),P(k genes),shots for 99%,seconds
p,,,,,,
1,-1.41792,0.99753,0.00251,1.0,1833,17.20245
2,-1.46709,0.99779,0.00330,1.0,1393,31.04303


## 3. Breast

In [15]:
br = results["breast"] = {}
br["adata"] = h.prepare(h.load_breast())
br["train"], br["test"] = h.split(br["adata"], test_size=TEST_SIZE)
print(br["adata"].n_obs, "cells |", len(br["train"]), "train |", len(br["test"]), "test")

515 cells | 412 train | 103 test


In [16]:
br["cands"] = {}
for pool in POOLS:
    br["cands"][pool] = [str(g) for g in h.candidates(br["adata"], br["train"], method=pool, n_boot=50)]
    print(f"{pool:12s}", br["cands"][pool])

markers_hvg  ['EPCAM', 'KRT8', 'KRT19', 'KRT18', 'ERBB2', 'CLU', 'TPM1', 'TFF3', 'HLA-DQA1', 'PTPRC', 'SAMSN1', 'GPR183', 'CD48', 'MX1', 'IGKC', 'BCL2A1']
wilcoxon     ['PTPRF', 'MYO6', 'ERBB3', 'ENAH', 'EPCAM', 'CD9', 'CTTN', 'FXYD3', 'RGS1', 'CD53', 'CD74', 'LAPTM5', 'HLA-DRB1', 'HLA-DPB1', 'HLA-DRB5', 'IFI16']
lasso        ['PTPRF', 'ERBB3', 'ESRP1', 'FXYD3', 'CTTN', 'RP11-206M11.7', 'DDR1', 'TFAP2C', 'HLA-DRB5', 'RGS1', 'CD53', 'FSTL1', 'IFI16', 'APOBEC3C', 'HLA-DRB1', 'ZEB2']
stable       ['PTPRF', 'MYO6', 'ERBB3', 'ENAH', 'EPCAM', 'CD9', 'FXYD3', 'KRT8', 'RGS1', 'CD53', 'CD74', 'LAPTM5', 'HLA-DRB1', 'IFI16', 'CD69', 'HLA-DPB1']


In [17]:
br["qubo"] = {}
for pool, cands in br["cands"].items():
    alpha, cv = h.choose_alpha_cv(br["adata"], br["train"], cands, K)
    Q, const, info = h.build_qubo(br["adata"], br["train"], cands, K, alpha, "auto")
    panel = h.best_panel(Q, const, cands, K)
    br["qubo"][pool] = {"Q": Q, "const": const, "info": info, "alpha": alpha, "panel": panel}
    print(f"{pool:12s} alpha={alpha:<5} lambda={info['lam']:.3f}  QUBO pick: {panel}")


markers_hvg  alpha=0.1   lambda=0.799  QUBO pick: ['EPCAM', 'KRT8']
wilcoxon     alpha=0.0   lambda=0.809  QUBO pick: ['PTPRF', 'CD9']
lasso        alpha=0.0   lambda=0.816  QUBO pick: ['PTPRF', 'TFAP2C']
stable       alpha=0.0   lambda=0.809  QUBO pick: ['PTPRF', 'CD9']


In [18]:
br["lasso"] = h.lasso_panel(br["adata"], br["train"], K)

rows = []
for pool, q in br["qubo"].items():
    cands, r = br["cands"][pool], q["info"]["r"]
    panels = {"classical: top 2": [cands[i] for i in np.argsort(-r)[:K]],
              "QUBO (exact)": q["panel"]}
    for method, p in panels.items():
        m, s = h.repeated_splits(br["adata"], p, test_size=TEST_SIZE)
        rows.append({"pool": pool, "method": method, "panel": ", ".join(map(str, p)),
                     "test": round(h.evaluate(br["adata"], p, br["train"], br["test"]), 3),
                     "10 splits": f"{m:.3f} ± {s:.3f}"})

m, s = h.repeated_splits(br["adata"], br["lasso"], test_size=TEST_SIZE)
rows.append({"pool": "all genes", "method": "classical: lasso", "panel": ", ".join(br["lasso"]),
             "test": round(h.evaluate(br["adata"], br["lasso"], br["train"], br["test"]), 3),
             "10 splits": f"{m:.3f} ± {s:.3f}"})

br["acc"] = pd.DataFrame(rows)
br["acc"]

,pool,method,panel,test,10 splits
0,markers_hvg,classical: top 2,"KRT8, EPCAM",0.935,0.898 ± 0.027
1,markers_hvg,QUBO (exact),"EPCAM, KRT8",0.935,0.898 ± 0.027
2,wilcoxon,classical: top 2,"PTPRF, CD9",0.882,0.892 ± 0.026
3,wilcoxon,QUBO (exact),"PTPRF, CD9",0.882,0.892 ± 0.026
4,lasso,classical: top 2,"TFAP2C, PTPRF",0.914,0.919 ± 0.018
5,lasso,QUBO (exact),"PTPRF, TFAP2C",0.914,0.919 ± 0.018
6,stable,classical: top 2,"PTPRF, CD9",0.882,0.892 ± 0.026
7,stable,QUBO (exact),"PTPRF, CD9",0.882,0.892 ± 0.026
8,all genes,classical: lasso,"TFAP2C, PTPRF",0.914,0.919 ± 0.018


In [19]:
q = br["qubo"]["stable"]
br["E"] = h.all_energies(q["Q"], q["const"])

br["qaoa_x"], _ = h.qaoa_ladder(br["E"], p_max=3, k=K)
br["qaoa_xy"], _ = h.qaoa_ladder(br["E"], p_max=2, mixer="xy", init="dicke", k=K)

print("X mixer");  display(br["qaoa_x"].round(5))
print("XY mixer"); display(br["qaoa_xy"].round(5))

X mixer


,avg energy,approx ratio,P(best panel),P(k genes),shots for 99%,seconds
p,,,,,,
1,-0.48944,0.99297,0.00287,0.33914,1603,7.44980
2,-0.98194,0.99628,0.00352,0.41148,1306,53.78566
3,-1.14520,0.99737,0.00381,0.43422,1208,81.29645


XY mixer


,avg energy,approx ratio,P(best panel),P(k genes),shots for 99%,seconds
p,,,,,,
1,-1.41800,0.99920,0.00956,1.0,480,18.99152
2,-1.42416,0.99925,0.01122,1.0,409,31.35188


In [20]:
import inspect
print(inspect.getsource(h._fit_score))

def _fit_score(Xtr, ytr, Xte, yte):
    # class_weight="balanced": both classes count equally during training,
    # matching the balanced-accuracy score used for grading
    model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, ytr)
    return balanced_accuracy_score(yte, model.predict(Xte))



In [21]:
print("balanced" in h._fit_score.__code__.co_consts)

True
